In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sherlock-holm-es-stories-plain-text/sherlock-holm.es_stories_plain-text_advs.txt


# Reading and Storing .txt Data

In [2]:
with open("/kaggle/input/sherlock-holm-es-stories-plain-text/sherlock-holm.es_stories_plain-text_advs.txt", "r", encoding="utf-8") as f:
    text = f.read()


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer

2026-01-26 21:03:14.390063: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769461394.833550      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769461394.976496      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769461396.147600      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769461396.147667      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769461396.147670      24 computation_placer.cc:177] computation placer alr

# Breaking into tokens - Tokenizing

In [4]:
tokenizer = Tokenizer()

In [5]:
tokenizer.fit_on_texts([text])

In [6]:
len(tokenizer.word_index)

8199

In [7]:
# tokenizer.word_counts

In [8]:
input_seq = []
for sentence in text.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(tokenized_sentence)):
        input_seq.append(tokenized_sentence[:i+1])
    

In [9]:
len(input_seq)

96314

In [10]:
max_length = max([len(x) for x in input_seq])

In [11]:
max_length

18

# Adding Padding to Sequences

In [12]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [13]:
padded_sequences = pad_sequences(input_seq, maxlen = max_length, padding = 'pre')

In [14]:
padded_sequences

array([[   0,    0,    0, ...,    0,    1, 1561],
       [   0,    0,    0, ...,    1, 1561,    5],
       [   0,    0,    0, ..., 1561,    5,  129],
       ...,
       [   0,    0,    0, ...,    1, 8198, 8199],
       [   0,    0,    0, ..., 8198, 8199, 3187],
       [   0,    0,    0, ..., 8199, 3187, 3186]], dtype=int32)

In [15]:
padded_sequences.shape

(96314, 18)

# Dividing into X and y

In [16]:
X = padded_sequences[:,:-1]
y = padded_sequences[:,-1]

In [17]:
X.shape

(96314, 17)

In [18]:
y.shape

(96314,)

# Changing y into categorical classes
In the above code, we are converting the output array into a suitable format for training a model, where each target word is represented as a binary vector.

In [19]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes=8200)

In [20]:
y.shape

(96314, 8200)

# Model Building and Training

In [21]:
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding

In [22]:
model = Sequential()

model.add(Embedding(8200, 100, input_shape=(17,)))
model.add(LSTM(150))
model.add(Dense(8200, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1769461417.474907      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1769461417.478833      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 17, 100)        │       820,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8200)           │     1,238,200 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,208,800 (8.43 MB)

 Trainable params: 2,208,800 (8.43 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [25]:
model.fit(X,y,epochs=40)

Epoch 1/40


I0000 00:00:1769461431.834863      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


3010/3010 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - accuracy: 0.0609 - loss: 6.5519
Epoch 2/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.1200 - loss: 5.5424
Epoch 3/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.1482 - loss: 5.1148
Epoch 4/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.1679 - loss: 4.7457
Epoch 5/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.1822 - loss: 4.4487
Epoch 6/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.2053 - loss: 4.1450
Epoch 7/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.2345 - loss: 3.8633
Epoch 8/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.2682 - loss: 3.5865
Epoch 9/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.3027 - loss: 3.3386
Epoch 10/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.3432 - loss: 3.1084
Epoch 11/40
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.3786 - loss: 2.9041
Epoch 12/40
3010/3010 ━━━━━━━━

In [26]:
import time
import numpy as np
text = "my name is"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=56, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
my name is helen
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
my name is helen stoner
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
my name is helen stoner and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
my name is helen stoner and i
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
my name is helen stoner and i am
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
my name is helen stoner and i am living
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
my name is helen stoner and i am living with
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
my name is helen stoner and i am living with my
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
my name is helen stoner and i am living with my stepfather
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
my name is helen stoner and i am living with my stepfather who
